# NB-25 — a confidence covariate for every attacked checkpoint

**Why.** Iso-WER matching equalises word error rate, but a targeted attack does
not exploit word error rate — it exploits how *confident* the model is. Two
checkpoints can agree on WER and differ in logit sharpness, and the paper has no
measurement of that. The review asked for a margin covariate, and noted
correctly that it costs one forward pass per checkpoint, not a re-run.

**What it computes,** on the same 100 utterances the attack uses, against the
**reference** transcript (no attack, no perturbation):

| quantity | what it says |
|---|---|
| `ctc_loss_ref` | mean CTC loss on the true transcript. Low = the model is sure of the right answer. |
| `margin_mean` | per frame, top log-probability minus the second. Averaged over frames. The direct measure of how contested each decision is. |
| `margin_p10` | the 10th percentile of that per-frame margin — the frames that are easiest to flip. |
| `blank_frac` | the share of frames whose argmax is the CTC blank. |
| `entropy_mean` | mean per-frame entropy of the output distribution. |

**How it is used.** Add `margin_mean` to the ANCOVA beside clean WER. If the
exposure coefficient survives, the accuracy control was adequate. If it absorbs
the exposure effect, then what looked like a data-volume effect was a confidence
effect, and the paper must say so.

**Cost.** One forward pass per checkpoint, 16 checkpoints, no gradients. Well
under one GPU-hour. Run it on a T4; it would also run on CPU, slowly.


## 1 · Every attacked checkpoint

In [ ]:
NOTEBOOK = "NB-25-margin-covariate"

# All 16 attacked conditions. Each needs its checkpoint dataset attached; any
# that is missing is reported and skipped rather than silently dropped.
QUEUE = [
    dict(tag="1h final",        ckpt_dataset="ckpt-xlsr-1h",        rung=1,   checkpoint="final"),
    dict(tag="1h run2",         ckpt_dataset="ckpt-xlsr-1h-traj",   rung=1,   checkpoint="final"),
    dict(tag="1h s1338",        ckpt_dataset="ckpt-xlsr-1h-s1338",  rung=1,   checkpoint="final"),
    dict(tag="1h s1339",        ckpt_dataset="ckpt-xlsr-1h-s1339",  rung=1,   checkpoint="final"),
    dict(tag="5h final",        ckpt_dataset="ckpt-xlsr-5h",        rung=5,   checkpoint="final"),
    dict(tag="5h s1338",        ckpt_dataset="ckpt-xlsr-5h-s1338",  rung=5,   checkpoint="final"),
    dict(tag="5h s1339",        ckpt_dataset="ckpt-xlsr-5h-s1339",  rung=5,   checkpoint="final"),
    dict(tag="25h final",       ckpt_dataset="ckpt-xlsr-25h",       rung=25,  checkpoint="final"),
    dict(tag="25h s1338",       ckpt_dataset="ckpt-xlsr-25h-s1338", rung=25,  checkpoint="final"),
    dict(tag="25h s1339",       ckpt_dataset="ckpt-xlsr-25h-s1339", rung=25,  checkpoint="final"),
    dict(tag="100h ms-1000",    ckpt_dataset="ckpt-xlsr-100h",      rung=100, checkpoint="milestone-1000"),
    dict(tag="100h ms-2000",    ckpt_dataset="ckpt-xlsr-100h",      rung=100, checkpoint="milestone-2000"),
    dict(tag="100h ms-3500",    ckpt_dataset="ckpt-xlsr-100h",      rung=100, checkpoint="milestone-3500"),
    dict(tag="100h s1338 ms-1000", ckpt_dataset="ckpt-xlsr-100h-s1338", rung=100, checkpoint="milestone-1000"),
    dict(tag="100h s1338 ms-2000", ckpt_dataset="ckpt-xlsr-100h-s1338", rung=100, checkpoint="milestone-2000"),
    dict(tag="100h s1338 ms-3500", ckpt_dataset="ckpt-xlsr-100h-s1338", rung=100, checkpoint="milestone-3500"),
]

# Exposure hours per condition, matching analysis/reanalysis.py COND.
HOURS = {1: 0.98, 5: 4.90, 25: 24.60}
MS_HOURS = {"milestone-1000": 38.80, "milestone-2000": 77.70, "milestone-3500": 98.02}

EDIT_TYPE, N_EVAL, BATCH_SIZE, SEED = "T5", 100, 8, 1337
USE_AMP = False      # no gradients here; fp32 keeps the margins exact
# Present only because the cloned core cells read them. Nothing is attacked.
EPS_LO, EPS_HI, BISECT_ROUNDS = 1e-4, 2e-3, 0
PROBE_STEPS = CONFIRM_STEPS = STALL_PATIENCE = CHECK_EVERY = 0
FIXED_EPS, LR, VALIDATE_BATCHING = None, 1e-4, False

print(f"{NOTEBOOK} | {len(QUEUE)} checkpoints | forward passes only, no attack")

## 2 · Boilerplate

In [ ]:
import os, hashlib
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # attacks optimise the INPUT: never DataParallel
import sys, json, time, random, re, glob, unicodedata, gc
import numpy as np, pandas as pd, torch
import torch.nn.functional as F
torch.backends.cudnn.benchmark = True

SEED = 1337
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
WORK, TEMP, DEVICE = "/kaggle/working", "/kaggle/temp", "cuda"
assert torch.cuda.is_available(), "Settings -> Accelerator -> GPU T4 x2"

# PROV holds only what is defined AT THIS POINT. In NB-05 the rung, edit type
# and bracket were module-level constants, so they could be read here; in a
# queue runner they are per-condition and are not bound yet. Reading them here
# is a NameError that fires three cells before anything expensive starts. The
# run-specific fields are added by the summary cell at the end.
# Project-wide Bengali normaliser. Defined HERE, in boilerplate, because the
# input-resolution cell asserts with it and in NB-05 it lived in the (later)
# target-construction cell -- which a queue runner does not have.
ZW = dict.fromkeys(map(ord, "\u200c\u200d"), None)
def norm_bn(s):
    return re.sub(r"\s+", " ", unicodedata.normalize("NFC", str(s)).translate(ZW)).strip()

PROV = {"notebook": NOTEBOOK, "seed": SEED,
        "gpu": torch.cuda.get_device_name(0), "torch": torch.__version__,
        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S")}
print(json.dumps(PROV, indent=2))

In [ ]:
# Internet is off in this session, so `pip install transformers==4.44.2` cannot
# reach the index. Measured twice in this project's own logs (NB-12 and NB-13,
# 2026-09-30): the cell spends ~202 s timing out, prints two ERROR lines, fails,
# and leaves the preinstalled version in place anyway. Pure waste.
#
# So we do not attempt the pin. We import what is already present and RECORD the
# version instead -- the pin's purpose was reproducibility, and a recorded
# version serves that better than a failed install does.
#
# If internet is ever turned ON, restore the pin: 4.44.2 is the version every
# figure measured before 2026-09-30 was produced on, and mixing versions across
# figures is exactly what the provenance record exists to catch.
#
# `jiwer` is deliberately NOT imported: it is not preinstalled on Kaggle, and
# the only place this notebook would use it is a FIXED_EPS clean-WER branch
# that cannot execute (FIXED_EPS is None here) and already guards itself.
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor
import transformers, librosa, numpy
print(f"transformers {transformers.__version__} | numpy {numpy.__version__} "
      f"| librosa {librosa.__version__}")
if transformers.__version__ != "4.44.2":
    print(f"  NOTE: this is NOT the project pin (4.44.2). Record "
          f"{transformers.__version__} against every number this run produces.")
PROV["transformers"] = transformers.__version__
PROV["numpy"] = numpy.__version__


## 3 · Inputs, eval set and frozen targets

In [ ]:
# ---------------------------------------------------------------------------
# Kaggle does not use one mount layout. Depending on the kernel you get
#     /kaggle/input/<name>                (older kernels)
#     /kaggle/input/datasets/<name>       (your own datasets, current)
#     /kaggle/input/competitions/<name>   (competition data)
# A flat glob only sees the wrapper dirs -- NB-01 failed with exactly this,
# reporting: Visible inputs: ['competitions', 'datasets']
# ---------------------------------------------------------------------------
inputs = sorted({p for pat in ("/kaggle/input/*", "/kaggle/input/*/*",
                               "/kaggle/input/*/*/*")
                 for p in glob.glob(pat) if os.path.isdir(p)})

SPLITS = next((p for p in inputs if "splits" in p.lower()), None)
assert SPLITS, ("Attach bn-asr-splits -- test.csv is needed to resolve audio paths "
                "for the 100 target utterances.\n"
                f"Visible inputs: {[os.path.basename(p) for p in inputs]}")
print("splits:", SPLITS)

# ---------------------------------------------------------------------------
# EVAL SET AND TARGETS: read, never regenerated.
#
# Task B's targets are published inside every eps50-*-n100 dataset as
# task_b_targets_n100.csv. Reading them makes this run byte-comparable by
# construction rather than by reproducing a generator. The hash is checked.
# ---------------------------------------------------------------------------
TGT_CSV = next((f"{p}/task_b_targets_n100.csv" for p in inputs
                if os.path.exists(f"{p}/task_b_targets_n100.csv")), None)
assert TGT_CSV, ("task_b_targets_n100.csv not found. Attach ONE Task B result "
                 "dataset, e.g. zoayriaabedin/eps50-rung1-final-t5-n100.\n"
                 f"Visible inputs: {[os.path.basename(p) for p in inputs]}")
TGT_SHA = hashlib.sha256(open(TGT_CSV, "rb").read()).hexdigest()
print("targets:", TGT_CSV)
print("targets sha256:", TGT_SHA)
assert TGT_SHA == "8711dc56202b33f90e769c9735231b0fa8cf27fcbc20151f5466bcd9cc2b1256", (
    "These are NOT the targets the six Task B runs used. The comparison against "
    "run 1's 75% would be meaningless.")
tg = pd.read_csv(TGT_CSV)
assert len(tg) == 100 and tg.utt_id.nunique() == 100

_full = pd.read_csv(f"{SPLITS}/test.csv")
# source_tokens / target_tokens are carried through: the results writer uses
# both, and eps50_per_token divides by target_tokens. Dropping them here would
# KeyError at save time, after the whole sweep had run.
ev = tg[["utt_id", "target_text", "source_tokens", "target_tokens"]].merge(
        _full, on="utt_id", how="left")
assert ev["path"].notna().all(), "some target utterances are absent from test.csv"
assert (ev.text.map(norm_bn).values == tg.text.map(norm_bn).values).all(), \
    "transcript mismatch between the target file and test.csv"
EV_CSV = TGT_CSV
PROV["attack_eval_csv"] = os.path.basename(TGT_CSV)
PROV["targets_sha256"] = TGT_SHA
PROV["fixed_eps"] = FIXED_EPS
PROV["bisect_rounds"] = BISECT_ROUNDS
# The checkpoint dataset and model_dir are PER CONDITION here, so they are not
# recorded once at the top: every result row carries its own `model_dir`, and the
# queue summary records the tags.
for _c in ("path","text","target_text","source_tokens","target_tokens","duration"):
    assert _c in ev.columns, f"ev is missing {_c}, which a later cell needs"
print(f"eval set: {len(ev)} utterances, manifest order (original 25 first)")

# ---------------------------------------------------------------------------
# Re-resolve the audio root, same hazard as NB-01: NB-00 baked in the mount
# point from ITS session. Here the whole attack set is only 25 files, so check
# every one of them rather than sampling.
# ---------------------------------------------------------------------------
def resolve_audio(df):
    p0 = str(df["path"].iloc[0])
    if os.path.exists(p0):
        return df, os.path.dirname(p0)
    stem, ext = os.path.basename(os.path.dirname(p0)), os.path.splitext(p0)[1]
    cands = [d for pat in (f"/kaggle/input/*/{stem}",
                           f"/kaggle/input/*/*/{stem}",
                           f"/kaggle/input/*/*/*/{stem}")
             for d in glob.glob(pat) if os.path.isdir(d)]
    assert cands, (f"Audio directory '{stem}' not found under /kaggle/input.\n"
                   "Attach: + Add Input -> Competitions -> 'bengaliai speech'.\n"
                   f"Visible inputs: {[os.path.basename(p) for p in inputs]}")
    df = df.copy()
    df["path"] = cands[0] + "/" + df["utt_id"].astype(str) + ext
    return df, cands[0]

ev, AUDIO_ROOT = resolve_audio(ev)
absent = [p for p in ev["path"] if not os.path.exists(p)]
assert not absent, f"{len(absent)}/{len(ev)} attack audio files missing, e.g.\n  {absent[0]}"
print(f"audio root: {AUDIO_ROOT}")
print(f"attack set: {len(ev)} utterances, all present")

## 4 · Front end, loss, PGD, bisection

Byte-identical to NB-05. These read `model`, `processor`, `BLANK_ID` and `DO_NORMALIZE` as globals at call time, which is what lets Stage 1 swap checkpoints without redefining anything.

In [ ]:
SR = 16000

def load_batch(rows):
    """Length-bucketed batch. Returns padded waveforms + a validity mask."""
    wavs = []
    for _, r in rows.iterrows():
        a, _ = librosa.load(r["path"], sr=SR, mono=True)
        a = np.asarray(a, np.float32)
        pk = np.abs(a).max()
        wavs.append(a / max(pk, 1.0))
    T = max(len(w) for w in wavs)
    x = torch.zeros(len(wavs), T, dtype=torch.float32)
    m = torch.zeros(len(wavs), T, dtype=torch.float32)
    for i, w in enumerate(wavs):
        x[i, :len(w)] = torch.from_numpy(w); m[i, :len(w)] = 1.0
    return x.to(DEVICE), m.to(DEVICE)


def frontend_b(x, mask):
    """Masked zero-mean/unit-variance. Computing stats over the PADDING would
    corrupt short utterances in a mixed-length batch."""
    if DO_NORMALIZE:
        n = mask.sum(-1, keepdim=True).clamp(min=1)
        mu = (x * mask).sum(-1, keepdim=True) / n
        var = (((x - mu) * mask) ** 2).sum(-1, keepdim=True) / n
        x = (x - mu) / torch.sqrt(var + 1e-7)
        x = x * mask
    return x


def logits_b(x, mask):
    return model(input_values=frontend_b(x, mask),
                 attention_mask=mask.long()).logits


def ctc_per_sample(x, mask, tgt_pad, tgt_len):
    """Per-sample CTC loss -> lets us mask out finished samples."""
    lg = logits_b(x, mask)
    logp = F.log_softmax(lg.float(), -1).transpose(0, 1)          # (L,B,V)
    in_len = model._get_feat_extract_output_lengths(mask.sum(-1).long()).long()
    in_len = in_len.clamp(max=logp.shape[0])
    return F.ctc_loss(logp, tgt_pad, in_len, tgt_len,
                      blank=BLANK_ID, zero_infinity=True, reduction="none")


def decode_b(x, mask):
    with torch.no_grad():
        ids = torch.argmax(logits_b(x, mask), -1)
    return [norm_bn(s) for s in processor.batch_decode(ids)]


def int16_rt(x):
    return torch.clamp((x * 32767.0).round(), -32768.0, 32767.0) / 32767.0

def snr_db_b(clean, adv, mask):
    n = (adv - clean) * mask
    s = clean * mask
    return (10 * torch.log10(s.pow(2).sum(-1) / (n.pow(2).sum(-1) + 1e-12))).cpu().numpy()

In [ ]:
def pgd_batch(x, mask, tgt_pad, tgt_len, targets, eps_vec,
              steps, delta_init=None, lr=LR):
    """eps_vec: (B,) per-sample L-inf budgets.
       Returns (success bool array, frozen deltas, steps_used)."""
    B = x.shape[0]
    eps_col = eps_vec.view(-1, 1)
    delta = (torch.zeros_like(x) if delta_init is None
             else torch.max(torch.min(delta_init.clone(), eps_col), -eps_col))
    delta.requires_grad_(True)
    opt = torch.optim.Adam([delta], lr=lr)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

    active = torch.ones(B, dtype=torch.bool, device=DEVICE)
    frozen = torch.zeros_like(x)
    success = np.zeros(B, dtype=bool)
    best_loss = torch.full((B,), float("inf"), device=DEVICE)
    stall = torch.zeros(B, dtype=torch.long, device=DEVICE)

    for step in range(steps):
        if not active.any():
            break
        x_adv = torch.clamp(x + delta, -1, 1) * mask
        with torch.amp.autocast("cuda", enabled=USE_AMP):
            per = ctc_per_sample(x_adv, mask, tgt_pad, tgt_len)
        loss = (per * active.float()).sum() / active.float().sum().clamp(min=1)

        opt.zero_grad(); scaler.scale(loss).backward()
        scaler.unscale_(opt)
        if delta.grad is not None:
            delta.grad[~active] = 0          # finished samples stop moving
        scaler.step(opt); scaler.update()
        with torch.no_grad():
            delta.data = torch.max(torch.min(delta.data, eps_col), -eps_col)

        with torch.no_grad():                # stall detection
            improved = per < best_loss - 1e-3
            best_loss = torch.minimum(best_loss, per.detach())
            stall = torch.where(improved, torch.zeros_like(stall), stall + 1)
            give_up = active & (stall > STALL_PATIENCE)
            active = active & ~give_up

        if (step + 1) % CHECK_EVERY == 0:
            hyp = decode_b(int16_rt(torch.clamp(x + delta, -1, 1)) * mask, mask)
            with torch.no_grad():
                for i in range(B):
                    if active[i] and hyp[i] == targets[i]:
                        frozen[i] = delta[i].detach()
                        success[i] = True
                        active[i] = False
    return success, frozen, step + 1

In [ ]:
def eps50_batch(x, mask, tgt_pad, tgt_len, targets, verbose=True):
    """Per-sample geometric bisection for the smallest budget that still works.

    ---------------------------------------------------------------------------
    BUG FIXED 2026-09-22 (found in the rung-1 T5 run, 40% censoring).

    The previous version ended with

        ok, best, _ = pgd_batch(..., hi, CONFIRM_STEPS, delta_init=best)

    which REASSIGNED `best` wholesale to pgd_batch's return value. That return
    is a `frozen` buffer holding zeros for any sample that did not succeed on
    that particular pass -- so one unlucky final pass wiped out a delta that had
    already been verified at a larger budget. Those samples were then written
    out as censored, with SNR 70-86 dB (the signature of delta == 0) even though
    the ceiling probe had already proved them attackable.

    In the rung-1 run that discarded ~6 of 25 real measurements. Worse, it does
    so at a RATE THAT VARIES BY MODEL, so comparing median eps50 across rungs
    would have been comparing differently-biased subsamples -- straight into
    RQ1's headline.

    Fix: keep a running `best_eps` / `best_delta` that is only ever updated on a
    VERIFIED success. Nothing overwrites a confirmed result. Censoring now means
    exactly one thing: the attack failed at every budget tried, ceiling included.

    Side benefit: eps50 becomes a guaranteed UPPER BOUND on the true minimum.
    Optimiser noise can only make it conservative, never wrong in an
    unpredictable direction -- which is the property you want in a metric you
    are about to compare across four models.
    ---------------------------------------------------------------------------
    """
    B = x.shape[0]
    lo = torch.full((B,), EPS_LO, device=DEVICE)
    hi = torch.full((B,), EPS_HI, device=DEVICE)
    best_eps   = torch.full((B,), float("nan"), device=DEVICE)   # verified only
    best_delta = torch.zeros_like(x)

    def record(okt, eps_vec, d):
        """Adopt a result only when it is both successful and better."""
        nonlocal best_eps, best_delta
        better = okt & (torch.isnan(best_eps) | (eps_vec < best_eps))
        best_eps   = torch.where(better, eps_vec, best_eps)
        best_delta = torch.where(better.view(-1, 1), d, best_delta)

    # Ceiling probe. Failure here is genuine right-censoring.
    ok, d, _ = pgd_batch(x, mask, tgt_pad, tgt_len, targets, hi, CONFIRM_STEPS)
    okt = torch.tensor(ok, device=DEVICE)
    record(okt, hi, d)
    feasible = ok.copy()
    if verbose:
        print(f"  ceiling probe eps={EPS_HI:.1e}: {feasible.sum()}/{B} feasible, "
              f"{(~feasible).sum()} censored")

    for rnd in range(BISECT_ROUNDS):
        mid = torch.sqrt(lo * hi)
        ok, d, _ = pgd_batch(x, mask, tgt_pad, tgt_len, targets, mid,
                             PROBE_STEPS, delta_init=best_delta)
        ok = ok & feasible
        okt = torch.tensor(ok, device=DEVICE)
        hi = torch.where(okt, mid, hi)
        lo = torch.where(okt, lo, mid)
        record(okt, mid, d)
        if verbose:
            print(f"  round {rnd+1}/{BISECT_ROUNDS}: {ok.sum()}/{B} succeeded "
                  f"| median eps {mid.median().item():.3e}")

    # Budget-consistency round: one more bisection at the FULL step budget, so
    # the reported boundary is measured at the budget eps50 is defined at rather
    # than at the cheaper probe budget. See RUNLOG for why this matters.
    #
    # Skipped in FIXED_EPS mode: there is no bracket to refine. The ceiling probe
    # already ran at CONFIRM_STEPS, so the single reported budget is measured at
    # the full budget and needs no second pass.
    if BISECT_ROUNDS == 0:
        eps       = best_eps.cpu().numpy()
        confirmed = ~np.isnan(eps)
        if verbose:
            print(f"  -> FIXED_EPS {EPS_HI:.1e}: {confirmed.sum()}/{B} succeeded, "
                  f"{(~confirmed).sum()} resisted")
        return eps, np.ones(B), confirmed, best_delta

    mid = torch.sqrt(lo * hi)
    ok, d, _ = pgd_batch(x, mask, tgt_pad, tgt_len, targets, mid,
                         CONFIRM_STEPS, delta_init=best_delta)
    ok = ok & feasible
    okt = torch.tensor(ok, device=DEVICE)
    record(okt, mid, d)
    if verbose:
        print(f"  budget-consistency round @ {CONFIRM_STEPS} steps: {ok.sum()}/{B}")

    eps       = best_eps.cpu().numpy()
    confirmed = ~np.isnan(eps)                 # censored == never succeeded
    width     = (hi / lo).cpu().numpy()
    if verbose:
        print(f"  -> {confirmed.sum()}/{B} measured, {(~confirmed).sum()} censored")
    return eps, width, confirmed, best_delta


## 5 · Per-condition loader

In [ ]:
def load_condition(cfg):
    """Resolve and load one checkpoint, rebinding the globals the attack closures read.

    The closures in the cells above look up `model`, `processor`, `BLANK_ID` and
    `DO_NORMALIZE` at CALL time, not definition time, so rebinding them here is
    all that is needed to point the whole machine at a different checkpoint.
    Every one of the four must be rebound: leaving a stale BLANK_ID or
    DO_NORMALIZE behind would silently attack the new model through the old
    model's decoding assumptions.
    """
    global model, processor, BLANK_ID, DO_NORMALIZE, MODEL_DIR, RUNG, CHECKPOINT
    ds = cfg["ckpt_dataset"]
    cands = [p for p in inputs if re.search(rf"{re.escape(ds)}$", os.path.basename(p))]
    assert len(cands) == 1, (
        f"Need exactly one input matching '{ds}', found {len(cands)}: "
        f"{[os.path.basename(c) for c in cands]}")
    ck = cands[0]
    if cfg["checkpoint"] == "final":
        MODEL_DIR = ck if os.path.exists(f"{ck}/config.json") else f"{ck}/final"
    else:
        MODEL_DIR = f"{ck}/milestones/step-{cfg['checkpoint'].split('-')[1]}"
    assert os.path.exists(f"{MODEL_DIR}/config.json"), f"no config.json under {MODEL_DIR}"

    processor = Wav2Vec2Processor.from_pretrained(MODEL_DIR)
    model = Wav2Vec2ForCTC.from_pretrained(MODEL_DIR).float().to(DEVICE).eval()
    for p in model.parameters():
        p.requires_grad_(False)
    model.config.apply_spec_augment = False
    DO_NORMALIZE = bool(getattr(processor.feature_extractor, "do_normalize", True))
    BLANK_ID = model.config.pad_token_id
    RUNG, CHECKPOINT = cfg["rung"], cfg["checkpoint"]
    print(f"  loaded {MODEL_DIR}")
    print(f"  do_normalize {DO_NORMALIZE} | blank {BLANK_ID} | vocab {model.config.vocab_size}")
    return MODEL_DIR


def unload():
    global model, processor
    try:
        del model, processor
    except NameError:
        pass
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

## 6 · The measurement

One forward pass per batch against the **reference** transcript. No gradients,
no perturbation, no decoding of an adversarial target. Everything is computed
inside `torch.no_grad()`, over valid frames only — the padded frames of a
mixed-length batch would otherwise drag every statistic toward whatever the
model emits on silence.

In [ ]:
@torch.no_grad()
def margins_for_batch(x, mask, ref_texts):
    """Per-utterance confidence statistics against the true transcript."""
    lg = logits_b(x, mask)                                  # (B, L, V)
    logp = F.log_softmax(lg.float(), -1)
    in_len = model._get_feat_extract_output_lengths(mask.sum(-1).long()).long()
    in_len = in_len.clamp(max=logp.shape[1])

    top2 = logp.topk(2, dim=-1).values                      # (B, L, 2)
    per_frame_margin = top2[..., 0] - top2[..., 1]          # (B, L)
    ent = -(logp.exp() * logp).sum(-1)                      # (B, L)
    argmax = logp.argmax(-1)                                # (B, L)

    tp = [torch.tensor(processor.tokenizer(t).input_ids, device=DEVICE)
          for t in ref_texts]
    tl = torch.tensor([len(t) for t in tp], device=DEVICE)
    tpad = torch.nn.utils.rnn.pad_sequence(tp, batch_first=True, padding_value=0)
    ctc = F.ctc_loss(logp.transpose(0, 1), tpad, in_len, tl,
                     blank=BLANK_ID, zero_infinity=True, reduction="none")

    out = []
    for i in range(x.shape[0]):
        L = int(in_len[i])                                  # valid frames only
        m = per_frame_margin[i, :L].cpu().numpy()
        out.append(dict(
            n_frames=L,
            ctc_loss_ref=float(ctc[i]),
            ctc_loss_per_frame=float(ctc[i]) / max(L, 1),
            margin_mean=float(m.mean()),
            margin_median=float(np.median(m)),
            margin_p10=float(np.percentile(m, 10)),
            entropy_mean=float(ent[i, :L].mean()),
            blank_frac=float((argmax[i, :L] == BLANK_ID).float().mean()),
        ))
    return out

## 7 · Sweep every checkpoint

In [ ]:
SESSION_START = time.time()
rows, missing = [], []

for qi, cfg in enumerate(QUEUE, 1):
    print(f"\n[{qi}/{len(QUEUE)}] {cfg['tag']}")
    try:
        load_condition(cfg)
    except AssertionError as e:
        print(f"  SKIPPED, checkpoint not attached: {e}")
        missing.append(cfg["tag"]); continue
    hrs = (MS_HOURS[cfg["checkpoint"]] if cfg["checkpoint"].startswith("milestone")
           else HOURS[cfg["rung"]])
    for i in range(0, len(ev), BATCH_SIZE):
        sub = ev.iloc[i:i+BATCH_SIZE]
        xb, mb = load_batch(sub)
        for (_, r), st in zip(sub.iterrows(), margins_for_batch(xb, mb, list(sub.text))):
            rows.append(dict(tag=cfg["tag"], rung=cfg["rung"],
                             checkpoint=cfg["checkpoint"], hours=hrs,
                             utt_id=r["utt_id"], duration=r["duration"],
                             model_dir=MODEL_DIR, **st))
    df = pd.DataFrame(rows)
    pd.DataFrame(rows).to_csv(f"{WORK}/margins_n{N_EVAL}.csv", index=False)
    this = df[df.tag == cfg["tag"]]
    print(f"  margin_mean {this.margin_mean.mean():6.3f} | "
          f"ctc/frame {this.ctc_loss_per_frame.mean():6.3f} | "
          f"blank {this.blank_frac.mean():.3f}")
    unload()

if missing:
    print("\n*** NOT MEASURED (checkpoint dataset not attached): ***")
    for t in missing:
        print("   ", t)
    print("    Attach them and re-run, or the ANCOVA loses those conditions.")
print(f"\n{len(rows)} rows over {len(QUEUE)-len(missing)} checkpoints "
      f"in {(time.time()-SESSION_START)/60:.1f} min")

## 8 · Per-checkpoint summary, and a first look at the confound

In [ ]:
df = pd.DataFrame(rows)
g = (df.groupby(["tag", "rung", "hours"])
       .agg(margin_mean=("margin_mean", "mean"),
            margin_p10=("margin_p10", "mean"),
            ctc_per_frame=("ctc_loss_per_frame", "mean"),
            entropy=("entropy_mean", "mean"),
            blank=("blank_frac", "mean"))
       .reset_index().sort_values("hours"))
print(g.to_string(index=False))
g.to_csv(f"{WORK}/margins_by_checkpoint.csv", index=False)

# Published clean WER and ASR, so the covariate can be read against both.
PUB = {"1h final": (0.6355, 75.0), "1h run2": (0.6251, 41.0),
       "1h s1338": (0.5905, 57.0), "1h s1339": (0.6240, 39.0),
       "5h final": (0.4233, 79.0), "5h s1338": (0.4152, 96.0),
       "5h s1339": (0.4187, 59.0), "25h final": (0.3114, 66.0),
       "25h s1338": (0.2987, 78.0), "25h s1339": (0.2907, 57.0),
       "100h ms-1000": (0.6943, 37.0), "100h ms-2000": (0.4752, 46.0),
       "100h ms-3500": (0.3449, 53.0), "100h s1338 ms-1000": (0.6955, 34.0),
       "100h s1338 ms-2000": (0.4960, 48.0), "100h s1338 ms-3500": (0.3495, 56.0)}

print("\n--- does the margin separate the iso-WER pairs? " + "-" * 25)
print("  If a pair is matched on WER but far apart on margin, clean WER was")
print("  never the right control and the review's point 4 stands.\n")
print(f"  {'pair':34s} {'clean WER':>18s} {'margin_mean':>20s}")
for lo, hi in [("1h final", "100h ms-1000"), ("5h final", "100h ms-2000"),
               ("25h final", "100h ms-3500")]:
    if lo in set(g.tag) and hi in set(g.tag):
        ml = float(g[g.tag == lo].margin_mean.iloc[0])
        mh = float(g[g.tag == hi].margin_mean.iloc[0])
        wl, wh = PUB[lo][0], PUB[hi][0]
        print(f"  {lo+' vs '+hi:34s} {wl:.4f} / {wh:.4f}   {ml:7.3f} / {mh:7.3f}"
              f"   gap {abs(ml-mh):.3f}")

print("\n--- correlations across checkpoints " + "-" * 37)
g2 = g.copy()
g2["wer"] = g2.tag.map(lambda t: PUB.get(t, (np.nan, np.nan))[0])
g2["asr"] = g2.tag.map(lambda t: PUB.get(t, (np.nan, np.nan))[1])
g2 = g2.dropna(subset=["wer", "asr"])
for a in ["margin_mean", "margin_p10", "ctc_per_frame", "entropy", "blank"]:
    print(f"  {a:14s} vs clean WER r = {g2[a].corr(g2.wer):+.3f}   "
          f"vs attack success r = {g2[a].corr(g2.asr):+.3f}")
print("\n  A covariate that correlates with attack success but NOT with clean WER")
print("  is exactly what the ANCOVA is missing. Add margin_mean to the fit in")
print("  analysis/reanalysis.py and see whether the exposure slope survives.")

PROV.update(notebook=NOTEBOOK, n_checkpoints=len(QUEUE)-len(missing),
            missing=missing, n_eval=int(len(ev)),
            session_minutes=round((time.time()-SESSION_START)/60, 1))
json.dump(PROV, open(f"{WORK}/_provenance.json", "w"), ensure_ascii=False,
          indent=2, default=str)
json.dump({"title": "margin-covariate", "id": "YOURUSERNAME/margin-covariate",
           "licenses": [{"name": "CC0-1.0"}]},
          open(f"{WORK}/dataset-metadata.json", "w"), indent=2)
for f in sorted(os.listdir(WORK)):
    print("   ", f)